In [2]:
# Sam Brown
# sam_brown@mines.edu
# Jul 25
# Goal: Preprocess the position data so we can use the inter event movement as a feature for our models

import sys
sys.path.append("/Users/sambrown04/Documents/SURF/whillans-surf/notebooks/SURF")

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

import my_lib.funcs


In [3]:
# Want to get information about what happens in between events so we will load in the time frames for each event and what stations were operational.
df_2010 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2010_2010Events2stas")
df_2011 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2011_2011Events2stas")
df_2012 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2012_2012Events2stas")
df_2013 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2013_2013Events2stas")
df_2014 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2014_2014Events2stas")
df_2015 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2015_2015Events2stas")
df_2016 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2016_2016Events2stas")
df_2017 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2017_2017Events2stas")
df_2018 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2018_2018Events2stas")
df_2019 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2019_2019Events2stas")

# One Large list
all_dfs = (
    df_2010 
    # + df_2011 
    # + df_2012 + df_2013 +
    # df_2014 + df_2015 + df_2016 + df_2017 + df_2018 + df_2019
)


In [4]:
dfs_sorted = sorted(all_dfs, key=lambda df: df['time'].iloc[0])

In [5]:
clean_df = my_lib.funcs.extract_event_features(dfs_sorted)

In [6]:
clean_df[0].head()

,station,pre-slip_area,slip_severity,peak_time,total_delta,start_time
0,la09x,13.285342,2.133253e-07,3420.0,0.091443,2010-01-01 15:25:30
1,slw1x,25.522392,1.232925e-06,3405.0,0.358676,2010-01-01 15:25:30


In [7]:
# Want to loop through and get the start_time end time for each event and the stations up during that time.
events_dict = []

for i, event in enumerate(dfs_sorted):
    start_time = event['time'].iloc[0]
    end_time = event['time'].iloc[-1]

    stations = clean_df[i]['station'].tolist()
    event = {
        'start_time': start_time,
        'end_time': end_time,
        'stations': stations
    }

    events_dict.append(event)

In [36]:
# Now to get the inter event positional data we need to load in by year and station

input_dir = "/Users/sambrown04/Documents/SURF/POS Data"
dfs_by_year = {}
years_to_include = [2010]

# loop over year folders
for year_folder in os.listdir(input_dir):
    year_path = os.path.join(input_dir, year_folder)

    if not year_folder.isdigit():
        continue

    year = int(year_folder)

    if year not in years_to_include:
        continue

    year = int(year_folder)
    dfs_by_year[year] = []

    # loop over CSVs in that year folder
    for file in os.listdir(year_path):
        if file.endswith('.csv'):
            file_path = os.path.join(year_path, file)
            try:
                df = pd.read_csv(file_path)
                dfs_by_year[year].append(df)
            except Exception as e:
                print(f"Failed to load {file_path}: {e}")

In [37]:
# For each event, have a dataframe that has the station as the columns and then a column for the datetime values
for year, df_list in dfs_by_year.items():
    for df in df_list:
        df['time'] = pd.to_datetime(df['time'])

In [38]:
# Now we would like to parse through the start times of the events and pull the data from the end time of the previous evt
# to the start time of the current event and make a dataframe
events_dfs = []
print("start")
# Loop through event start times defined above
for i, event in enumerate(events_dict):
    if i == 0: # we are looking back so skip first
        continue
        
    print(f"Loading Event {i}")
    
    # Define the interval between events and the stations up
    end_interval = datetime.strptime(events_dict[i]['start_time'], '%Y-%m-%d %H:%M:%S')
    start_interval = datetime.strptime(events_dict[i-1]['end_time'], '%Y-%m-%d %H:%M:%S')
    stations = events_dict[i]['stations']
    
    year = end_interval.year

    if year not in dfs_by_year:
        print(f"Missing data for year {year}")
        continue

    
    inter_df = pd.DataFrame()

    # Loop through stations in the year and add the correct time frame to the inter_df
    for df in dfs_by_year[year]:
        station = df['station'].iloc[0]
        # Select data during the interval
        mask = (df['time'] >= start_interval) & (df['time'] <= end_interval)
        filtered_df = df.loc[mask]
        
        xdat = filtered_df['x'].reset_index(drop=True)
        ydat = filtered_df['y'].reset_index(drop=True)
        zdat = filtered_df['elevation'].reset_index(drop=True)
        time = filtered_df['time'].reset_index(drop=True)

        partial_df = pd.DataFrame({
        f'{station}_x': xdat,
        f'{station}_y': ydat,
        f'{station}_z': zdat,
        "time": time
        })

        # Combine all the stations available
        inter_df = pd.concat([inter_df, partial_df], axis=1)

    events_dfs.append(inter_df)
    print(f"Event {i} DataFrame shape: {inter_df.shape}")
    

    
    

start
Loading Event 1
Event 1 DataFrame shape: (1522, 48)
Loading Event 2
Event 2 DataFrame shape: (3632, 48)
Loading Event 3
Event 3 DataFrame shape: (4982, 48)
Loading Event 4
Event 4 DataFrame shape: (1222, 48)
Loading Event 5
Event 5 DataFrame shape: (3152, 48)
Loading Event 6
Event 6 DataFrame shape: (1682, 48)
Loading Event 7
Event 7 DataFrame shape: (2902, 48)
Loading Event 8
Event 8 DataFrame shape: (2262, 48)
Loading Event 9
Event 9 DataFrame shape: (2582, 48)
Loading Event 10
Event 10 DataFrame shape: (5342, 48)
Loading Event 11
Event 11 DataFrame shape: (3782, 48)
Loading Event 12
Event 12 DataFrame shape: (1442, 48)
Loading Event 13
Event 13 DataFrame shape: (3182, 48)
Loading Event 14
Event 14 DataFrame shape: (1562, 48)
Loading Event 15
Event 15 DataFrame shape: (3422, 48)
Loading Event 16
Event 16 DataFrame shape: (1612, 48)
Loading Event 17
Event 17 DataFrame shape: (3432, 48)
Loading Event 18
Event 18 DataFrame shape: (1632, 48)
Loading Event 19
Event 19 DataFrame shap

In [ ]:
events_dfs[2].head(40)

In [ ]:
len(events_dfs)